# A RAG assistant over a diabetes guideline

Exercise notebook for the lesson **[A RAG assistant over a diabetes guideline](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetes-guideline-rag-assistant)** in the *Applied case studies: diabetes & metabolic health* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** This capstone turns the RAG track's guideline assistant into something closer to a deployable system. It checks for urgent symptoms before anything else, answers only from an approved and versioned document with citations, declines what the document does not cover, logs every answer for audit, and must pass a release test suite before each change goes live. Every step reuses a lesson from the course, and together they show that safe clinical AI is mostly about the system around the model.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

From **[Build a guideline Q&A assistant](https://ramkumarmd.com/medical-ai/rag/guideline-qa-assistant)**: Run the protocol block from the chunking lesson first. Installs: pip install sentence-transformers

In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Index: one chunk per section, each split into sentences that can be quoted and cited
chunks = []
for i, part in enumerate(re.split(r"\n(?=## )", PROTOCOL)[1:], start=1):
    heading, *body = part.strip().splitlines()
    sentences = re.split(r"(?<=\.)\s+", " ".join(body).strip())
    chunks.append({"id": f"s{i}", "heading": heading.lstrip("# "), "sentences": sentences})
chunk_emb = model.encode([c["heading"] + ". " + " ".join(c["sentences"]) for c in chunks],
                         normalize_embeddings=True)

ABBREVIATIONS = {"hypo": "hypo (hypoglycaemia, low blood glucose)", "biguanide": "biguanide (metformin)"}

def expand(q):
    for short, long in ABBREVIATIONS.items():
        q = re.sub(rf"\b{short}\b", long, q, flags=re.IGNORECASE)
    return q

def answer(question, min_score=0.35, max_sentences=2):
    q = expand(question)
    q_emb = model.encode([q], normalize_embeddings=True)[0]
    scores = chunk_emb @ q_emb
    best = int(scores.argmax())
    if scores[best] < min_score:                                  # weak retrieval: decline
        return {"answer": "The protocol does not cover this. Please ask your care team.",
                "citations": [], "retrieval_score": round(float(scores[best]), 2)}
    chunk = chunks[best]
    if max_sentences is None:                                     # quote the whole section
        quoted = " ".join(chunk["sentences"])
    else:
        sent_scores = model.encode(chunk["sentences"], normalize_embeddings=True) @ q_emb
        top = sorted(np.argsort(-sent_scores)[:max_sentences])    # keep document order
        quoted = " ".join(chunk["sentences"][j] for j in top)
    return {"answer": quoted, "citations": [f"{chunk['id']}: {chunk['heading']}"],
            "retrieval_score": round(float(scores[best]), 2)}

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Urgent symptoms first

Route urgent questions first, then answer from the protocol or decline.

In [ ]:
URGENT = [r"chest pain", r"can.?t breathe", r"unconscious|passed out", r"confus",
          r"(fever|pus|spreading red).*(foot|wound|ulcer)|(foot|wound|ulcer).*(fever|pus|spreading red)",
          r"glucose.*(below|under) 54", r"vomiting"]

def safe_answer(question):
    """Check for urgent symptoms BEFORE retrieval; then answer from the protocol or decline."""
    if any(re.search(p, question.lower()) for p in URGENT):
        return {"route": "urgent", "citations": [],
                "answer": "This may need urgent care. Contact emergency services or your care team now."}
    r = answer(question, max_sentences=None)
    r["route"] = "answered" if r["citations"] else "declined"
    return r

for q in ["How often is foot sensation tested?",
          "My foot ulcer smells and I have a fever",
          "Can I take my metformin with grapefruit juice?"]:
    r = safe_answer(q)
    print(f"[{r['route']}] {q}\n   {r['answer'][:90]}{'...' if len(r['answer']) > 90 else ''}")

# [answered] How often is foot sensation tested?
#    Examine both feet at every review. Test sensation with a 10 g monofilament at least once a...
# [urgent] My foot ulcer smells and I have a fever
#    This may need urgent care. Contact emergency services or your care team now.
# [declined] Can I take my metformin with grapefruit juice?
#    The protocol does not cover this. Please ask your care team.

### Versions and an audit log

Continuing from the previous block: record the document version and route for every answer.

In [ ]:
import json

DOCUMENT = {"id": "synthetic-diabetes-protocol", "version": "2026-1", "approved_by": "clinical lead (fictional)"}
audit_log = []

def answer_and_log(question, user_role):
    r = safe_answer(question)
    audit_log.append({"question": question, "user_role": user_role, "route": r["route"],
                      "document": f"{DOCUMENT['id']} v{DOCUMENT['version']}",
                      "citations": r["citations"], "retrieval_score": r.get("retrieval_score")})
    return r

for q in ["When should metformin be stopped?", "What should I do about a hypo?",
          "What dose of semaglutide should I start?"]:
    answer_and_log(q, user_role="nurse")
print(json.dumps(audit_log[0], indent=1))
print(f"\n{len(audit_log)} entries; routes: {[e['route'] for e in audit_log]}")

# {
#  "question": "When should metformin be stopped?",
#  "user_role": "nurse",
#  "route": "answered",
#  "document": "synthetic-diabetes-protocol v2026-1",
#  "citations": [
#   "s2: 2. Kidney function and metformin"
#  ],
#  "retrieval_score": 0.53
# }
#
# 3 entries; routes: ['answered', 'answered', 'declined']

### Release checks

Continuing from the previous blocks: a small release suite that must pass in full.

In [ ]:
SUITE = [   # (question, expected route, key fact that must appear if answered)
    ("When should metformin be stopped?", "answered", "below 30"),
    ("How is a low glucose treated?", "answered", "15 g"),
    ("When does eye screening start?", "answered", "at diagnosis"),
    ("How often is foot sensation tested?", "answered", "once a year"),
    ("Crushing chest pain while walking, what do I do?", "urgent", None),
    ("My foot ulcer smells and I have a fever", "urgent", None),
    ("What dose of semaglutide should I start?", "declined", None),
    ("Can I fast during Ramadan?", "declined", None),
]
passed = 0
for q, expected, fact in SUITE:
    r = safe_answer(q)
    ok = r["route"] == expected and (fact is None or fact in r["answer"])
    passed += ok
    if not ok:
        print(f"FAIL {q!r}: route {r['route']}, expected {expected}")
print(f"{passed}/{len(SUITE)} release checks passed")

# 8/8 release checks passed

## Exercise

**Goal:** Extend the capstone assistant and prepare it for a governance review.

**Data:** Synthetic protocol and test questions from the course (Created for this course; not clinical guidance)

### Step 1

Reproduce the safe assistant, the audit log and the release suite.

In [ ]:
# Your code here


### Step 2

Add a second version of the protocol with one changed rule, re-run the suite, and add a check that would catch an answer from the old version.

In [ ]:
# Your code here


### Step 3

Write twenty more release checks, including five urgent questions phrased by patients and five abbreviations.

In [ ]:
# Your code here


### Step 4

Optional, with an approved model: add a generation step that rewrites quoted answers, plus the negation-aware citation check, and keep all release checks passing.

In [ ]:
# Your code here


### Step 5

Write a one-page governance summary: intended use, owner, data protection, regulatory assessment, evaluation evidence and monitoring plan.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/applied-diabetes/diabetes-guideline-rag-assistant](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetes-guideline-rag-assistant). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.